In [0]:
# 1. Cài đặt thư viện cần thiết
%pip install yfinance

import yfinance as yf
from pyspark.sql.functions import lit, current_timestamp, col
import pandas as pd
from datetime import datetime

# 2. Danh sách 10 mã cổ phiếu lớn (Việt Nam & Mỹ) theo yêu cầu
# Việt Nam: FPT, VCB (Vietcombank), HPG (Hòa Phát)
# Mỹ: AAPL (Apple), TSLA (Tesla), MSFT (Microsoft), GOOGL (Google), AMZN (Amazon), META, NVDA
symbols = ["FPT.VN", "VCB.VN", "HPG.VN", "AAPL", "TSLA", "MSFT", "GOOGL", "AMZN", "META", "NVDA"]

print(f"Bắt đầu tiến trình Ingestion cho {len(symbols)} mã cổ phiếu...")

all_data = []

# 3. Thu thập dữ liệu từ Yahoo Finance
for symbol in symbols:
    try:
        ticker = yf.Ticker(symbol)
        # Lấy dữ liệu 1 năm gần nhất
        df_history = ticker.history(period="1y")
        
        if df_history.empty:
            print(f"Cảnh báo: Không có dữ liệu cho mã {symbol}")
            continue
            
        df_history = df_history.reset_index()
        df_history['symbol'] = symbol
        
        # Gán loại tiền tệ (VND cho mã .VN, USD cho mã còn lại)
        df_history['currency'] = "VND" if ".VN" in symbol else "USD"
        
        all_data.append(df_history)
        print(f"Đã tải xong: {symbol}")
    except Exception as e:
        print(f"Lỗi khi tải mã {symbol}: {str(e)}")

# 4. Gộp dữ liệu bằng Pandas
final_pdf = pd.concat(all_data)

# 5. Chuẩn hóa tên cột để khớp với bảng SQL Bronze
final_pdf = final_pdf.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume",
    "Dividends": "adj_close", # Sử dụng cột Dividends làm dữ liệu thô cho adj_close
    "Date": "date"
})

# 6. Chuyển sang Spark DataFrame và ép kiểu dữ liệu (Cực kỳ quan trọng)
spark_df = spark.createDataFrame(final_pdf)

# Xử lý kiểu dữ liệu để tránh lỗi [DELTA_FAILED_TO_MERGE_FIELDS]
spark_df = spark_df.withColumn("date", col("date").cast("date")) \
                   .withColumn("open", col("open").cast("double")) \
                   .withColumn("high", col("high").cast("double")) \
                   .withColumn("low", col("low").cast("double")) \
                   .withColumn("close", col("close").cast("double")) \
                   .withColumn("adj_close", col("adj_close").cast("double")) \
                   .withColumn("volume", col("volume").cast("long")) \
                   .withColumn("ingested_at", current_timestamp())

# 7. Chỉ chọn các cột cần thiết để ghi vào bảng Bronze
# (Lưu ý: Nếu bảng SQL của bạn chưa có cột 'currency', Spark sẽ tự thêm nhờ option mergeSchema)
final_columns = ["symbol", "open", "high", "low", "close", "volume", "adj_close", "date", "ingested_at"]
spark_df = spark_df.select(*final_columns)

# 8. Ghi dữ liệu vào Delta Table (Lớp Bronze)
target_table = "workspace.stock_db.bronze_stock_prices"

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)

print(f"--- THÀNH CÔNG: Đã nạp dữ liệu vào {target_table} ---")

# 9. Hiển thị kết quả kiểm tra
display(spark.sql(f"SELECT symbol, date, close, ingested_at FROM {target_table} ORDER BY date DESC LIMIT 20"))